<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/the-ai-engineer/blob/main/capstones/week03_transformers/week03_tiny_transformer.ipynb)

# A Tiny Transformer, From Scratch, on FOMC Statements and Minutes

**TAE Week 3 Capstone** · Francisco (FranQuant)

The Fed just changed chairs, and its communication style with it — shorter statements,
no forward guidance. Before that voice fully changes, this notebook trains a small
decoder-only transformer, built from scratch in PyTorch, to learn the outgoing one:
FOMC statements and minutes from 2010 through Chair Powell's final meeting in April 2026.
Attention, causal masking, multi-head attention, and the training loop are all
implemented here directly — no transformer libraries.

## Setup

In [ ]:
!nvidia-smi || true
!pip -q install tqdm


In [ ]:
import math, json, time, hashlib
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

plt.style.use('seaborn-v0_8')
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"device: {DEVICE}")


## Configuration

In [ ]:
SEED = 1
torch.manual_seed(SEED)
np.random.seed(SEED)

@dataclass
class ModelConfig:
    vocab_size: int = None
    d_model: int = 256
    num_heads: int = 8
    num_layers: int = 6
    d_ff: int = 1024
    block_size: int = 256
    dropout: float = 0.1

@dataclass
class TrainConfig:
    batch_size: int = 64
    lr: float = 3e-4
    min_lr: float = 3e-5
    warmup_iters: int = 200
    weight_decay: float = 0.01
    grad_clip: float = 1.0
    max_steps: int = 6000
    eval_interval: int = 300
    eval_iters: int = 50

CORPUS_PATH = Path("fomc_training_corpus.txt")
CKPT_DIR = Path("checkpoints"); CKPT_DIR.mkdir(exist_ok=True)
CKPT_PATH = CKPT_DIR / "tiny_transformer_best.pt"
RUN_DIR = Path("runs"); RUN_DIR.mkdir(exist_ok=True)

# Set TRAIN = False to skip training and validate from an existing checkpoint.
TRAIN = True


## Data

Character-level, a 90/10 train/validation split, and fixed-length context sampling.

In [ ]:
text = CORPUS_PATH.read_text(encoding="utf-8")
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n_split = int(0.9 * len(data))
train_data, val_data = data[:n_split], data[n_split:]
print(f"corpus: {len(text):,} chars | vocab: {vocab_size} | train: {len(train_data):,} | val: {len(val_data):,}")

model_cfg = ModelConfig(vocab_size=vocab_size)
train_cfg = TrainConfig()

def get_batch(split, block_size, batch_size, device=DEVICE):
    d = train_data if split == "train" else val_data
    ix = torch.randint(0, len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+1+block_size] for i in ix])
    return x.to(device), y.to(device)


## Scaled Dot-Product Attention

$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    S = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        S = S.masked_fill(mask == 0, float("-inf"))
    S = S - S.max(dim=-1, keepdim=True).values
    A = torch.softmax(S, dim=-1)
    return A @ V, S, A


A quick check against a hand-computable example: three tokens, 2D vectors.

In [ ]:
Qe = torch.tensor([[1.,0],[0,1],[1,1]])
Ke = torch.tensor([[1.,0],[1,1],[0,1]])
Ve = torch.tensor([[1.,0],[0,2],[3,1]])
Y, S, A = scaled_dot_product_attention(Qe, Ke, Ve)
assert torch.allclose(A.sum(dim=-1), torch.ones(3))
assert torch.allclose(Y, torch.tensor([[0.994440, 1.0], [1.401112, 1.203336], [0.993020, 1.255235]]), atol=1e-5)
print("attention weights sum to 1, output matches hand computation:")
print(Y)


With a causal mask, the first token has exactly one valid key -- itself -- so its attention row is forced to `[1, 0, 0]` no matter what the key vectors are.

In [ ]:
causal_mask = torch.tril(torch.ones(3, 3))
_, _, A_m = scaled_dot_product_attention(Qe, Ke, Ve, mask=causal_mask)
assert torch.allclose(A_m[0], torch.tensor([1., 0., 0.]))

fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
axes[0].imshow(A.detach(), cmap="Blues", vmin=0, vmax=1); axes[0].set_title("unmasked")
axes[1].imshow(A_m.detach(), cmap="Blues", vmin=0, vmax=1); axes[1].set_title("causal")
for ax in axes: ax.set_xticks(range(3)); ax.set_yticks(range(3))
fig.tight_layout(); plt.show()


## Self-Attention

Wraps attention with learned query/key/value projections.

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, d_k=None, d_v=None, causal=True):
        super().__init__()
        d_k, d_v = d_k or d_model, d_v or d_model
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_v, bias=False)
        self.causal = causal

    def forward(self, x):
        B, T, _ = x.shape
        Q, K, V = self.W_Q(x), self.W_K(x), self.W_V(x)
        mask = torch.tril(torch.ones(T, T, device=x.device)) if self.causal else None
        y, _, _ = scaled_dot_product_attention(Q, K, V, mask=mask)
        return y


Sanity check: zero out the query/key projections so every attention score is equal, and set the value projection to identity. Attention then has to be uniform, so the layer collapses to a plain average over positions.

In [ ]:
sa = SelfAttention(d_model=3, causal=False)
with torch.no_grad():
    sa.W_Q.weight.zero_(); sa.W_K.weight.zero_(); sa.W_V.weight.copy_(torch.eye(3))
x_probe = torch.randn(1, 4, 3)
assert torch.allclose(sa(x_probe), x_probe.mean(dim=1, keepdim=True).expand_as(x_probe), atol=1e-5)
print("reduces to a positional average, as expected")


## Multi-Head Attention and Transformer Blocks

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0, causal=True):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads, self.d_head = num_heads, d_model // num_heads
        self.causal = causal
        self.proj_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj_out = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape
        qkv = self.proj_qkv(x).view(B, T, 3, self.num_heads, self.d_head)
        Q, K, V = (t.transpose(1, 2) for t in qkv.unbind(dim=2))
        mask = torch.tril(torch.ones(T, T, device=x.device)) if self.causal else None
        Y, _, _ = scaled_dot_product_attention(Q, K, V, mask=mask)
        Y = Y.transpose(1, 2).contiguous().view(B, T, D)
        return self.dropout(self.proj_out(Y))


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(),
                                  nn.Linear(d_ff, d_model), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.0, causal=True):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads, dropout=dropout, causal=causal)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout=dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


With identical parameters across heads, multi-head attention should match single-head attention exactly -- checked here at `num_heads=1`.

In [ ]:
sa_ref = SelfAttention(d_model=4, causal=True)
mha_ref = MultiHeadAttention(d_model=4, num_heads=1, causal=True)
with torch.no_grad():
    mha_ref.proj_qkv.weight.copy_(torch.cat([sa_ref.W_Q.weight, sa_ref.W_K.weight, sa_ref.W_V.weight], dim=0))
    mha_ref.proj_out.weight.copy_(torch.eye(4))
x2 = torch.randn(1, 3, 4)
assert torch.allclose(mha_ref(x2), sa_ref(x2), atol=1e-5)
print("multi-head (1 head) matches single-head attention")


In [ ]:
tiny_block = TransformerBlock(d_model=4, num_heads=2, d_ff=8, causal=True)
x_tiny = torch.randn(1, 3, 4)
out_tiny = tiny_block(x_tiny)
assert out_tiny.shape == (1, 3, 4) and torch.isfinite(out_tiny).all()
print(f"block forward pass ok, shape {tuple(out_tiny.shape)}")


## Positional Encoding

Standard sinusoidal encoding, added to the token embeddings.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.shape[1], :]


## The Model

Token embedding + positional encoding, a stack of transformer blocks, a final LayerNorm, and an output projection tied to the input embedding.

In [ ]:
class TinyTransformerLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.block_size = cfg.block_size
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_enc = PositionalEncoding(cfg.d_model, max_len=cfg.block_size)
        self.blocks = nn.ModuleList([
            TransformerBlock(cfg.d_model, cfg.num_heads, cfg.d_ff, cfg.dropout, causal=True)
            for _ in range(cfg.num_layers)
        ])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx, targets=None):
        z = self.pos_enc(self.tok_emb(idx))
        for blk in self.blocks:
            z = blk(z)
        z = self.ln_f(z)
        logits = self.head(z)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, greedy=False):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-8)
            next_id = logits.argmax(dim=-1, keepdim=True) if greedy else \
                torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

model = TinyTransformerLM(model_cfg).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params:,}")


Before trusting this on the real corpus: can it even overfit a trivial repeating pattern?

In [ ]:
torch.manual_seed(SEED)
toy_text = "AB" * 200
toy_stoi = {c: i for i, c in enumerate(sorted(set(toy_text)))}
toy_data = torch.tensor([toy_stoi[c] for c in toy_text])
def toy_batch(bs=8, n=16):
    ix = torch.randint(0, len(toy_data) - bs - 1, (n,))
    return (torch.stack([toy_data[i:i+bs] for i in ix]),
            torch.stack([toy_data[i+1:i+1+bs] for i in ix]))

toy_cfg = ModelConfig(vocab_size=len(toy_stoi), d_model=16, num_heads=2, num_layers=2, d_ff=32, block_size=8, dropout=0.0)
toy_model = TinyTransformerLM(toy_cfg)
toy_opt = torch.optim.Adam(toy_model.parameters(), lr=3e-3)
toy_loss = None
for step in range(300):
    xb, yb = toy_batch()
    _, toy_loss = toy_model(xb, yb)
    toy_opt.zero_grad(); toy_loss.backward(); toy_opt.step()
assert toy_loss.item() < 0.05, toy_loss.item()
print(f"overfits cleanly (final loss {toy_loss.item():.4f})")


## Run Record

In [ ]:
def save_run(cfg_model, cfg_train, train_loss, val_loss, ckpt_path, sha256, n_params):
    rec = {"seed": SEED, "model_config": asdict(cfg_model), "train_config": asdict(cfg_train),
           "train_loss": train_loss, "val_loss": val_loss, "checkpoint_path": str(ckpt_path),
           "corpus_sha256": sha256, "n_params": n_params, "timestamp": datetime.now().isoformat()}
    path = RUN_DIR / f"run_{datetime.now().strftime('%Y%m%dT%H%M%S')}.json"
    path.write_text(json.dumps(rec, indent=2), encoding="utf-8")
    return path


## Training

Adam with a cosine learning-rate schedule and a short warmup, weight decay, and gradient clipping. `TRAIN` controls whether training runs (set in Configuration). to loading the committed checkpoint, so a fresh runtime can verify everything in seconds without retraining.

In [ ]:
@torch.no_grad()
def evaluate(model, split, block_size, batch_size, iters):
    model.eval()
    losses = [model(*get_batch(split, block_size, batch_size))[1].item() for _ in range(iters)]
    model.train()
    return sum(losses) / len(losses)

def lr_at(step, cfg: TrainConfig):
    if step < cfg.warmup_iters:
        return cfg.lr * step / max(1, cfg.warmup_iters)
    progress = (step - cfg.warmup_iters) / max(1, cfg.max_steps - cfg.warmup_iters)
    return cfg.min_lr + 0.5 * (cfg.lr - cfg.min_lr) * (1 + math.cos(math.pi * progress))

corpus_sha256 = hashlib.sha256(text.encode("utf-8")).hexdigest()


In [ ]:
if TRAIN:
    opt = torch.optim.AdamW(model.parameters(), lr=train_cfg.lr, weight_decay=train_cfg.weight_decay)
    history = {"step": [], "train_loss": [], "val_loss": []}
    pbar = tqdm(range(1, train_cfg.max_steps + 1))
    for step in pbar:
        lr = lr_at(step, train_cfg)
        for g in opt.param_groups: g["lr"] = lr
        xb, yb = get_batch("train", model_cfg.block_size, train_cfg.batch_size)
        _, loss = model(xb, yb)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip)
        opt.step()
        if step % train_cfg.eval_interval == 0 or step == train_cfg.max_steps:
            tr = evaluate(model, "train", model_cfg.block_size, train_cfg.batch_size, train_cfg.eval_iters)
            va = evaluate(model, "val", model_cfg.block_size, train_cfg.batch_size, train_cfg.eval_iters)
            history["step"].append(step); history["train_loss"].append(tr); history["val_loss"].append(va)
            pbar.set_postfix(train=f"{tr:.3f}", val=f"{va:.3f}", lr=f"{lr:.1e}")


In [ ]:
if TRAIN:
    torch.save({"model_state_dict": model.state_dict(), "model_config": asdict(model_cfg)}, CKPT_PATH)
    run_path = save_run(model_cfg, train_cfg, history["train_loss"][-1],
                        history["val_loss"][-1], CKPT_PATH, corpus_sha256, n_params)
    print(f"checkpoint: {CKPT_PATH}  |  run record: {run_path}")

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(history["step"], history["train_loss"], label="train")
    ax.plot(history["step"], history["val_loss"], label="val")
    ax.set(xlabel="step", ylabel="loss", title="Training and validation loss")
    ax.legend(); fig.tight_layout(); plt.show()
else:
    print("TRAIN is False -- loading the committed checkpoint instead.")


In [ ]:
if CKPT_PATH.exists():
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    loaded_cfg = ModelConfig(**ckpt["model_config"])
    model = TinyTransformerLM(loaded_cfg).to(DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    val_loss_reloaded = evaluate(model, "val", loaded_cfg.block_size, 32, 20)
    print(f"checkpoint loaded, val loss {val_loss_reloaded:.4f}")
else:
    print(f"no checkpoint at {CKPT_PATH} yet -- set TRAIN = True above and rerun")
    val_loss_reloaded = None


## Sampling

Greedy decoding, and temperature sampling at three settings.

In [ ]:
def sample(model, prompt, max_new_tokens=200, temperature=1.0, greedy=False):
    model.eval()
    unknown = sorted(set(prompt) - set(stoi))
    if unknown:
        raise ValueError(f"prompt has character(s) outside the training vocabulary: {unknown!r}")
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)
    return decode(model.generate(idx, max_new_tokens, temperature=temperature, greedy=greedy)[0].tolist())

prompts = ["<|fomc_statement|>\ndate: 2026-", "The Committee decided to ", "<|fomc_minutes|>\ndate: 2026-"]
prompts = [p for p in prompts if not (set(p) - set(stoi))]

if CKPT_PATH.exists():
    for p in prompts:
        print(f"--- {p!r} ---")
        print("greedy :", sample(model, p, 150, greedy=True))
        for t in (0.7, 1.0, 1.5):
            print(f"t={t:.1f}  :", sample(model, p, 150, temperature=t))
        print()


## Optional: Attention on a Real Prompt

One extra, clearly optional figure: what the trained model actually attends to, on a real corpus snippet rather than the toy example above.

In [ ]:
if CKPT_PATH.exists():
    probe_text = decode(train_data[1000:1030].tolist())
    probe_idx = torch.tensor([encode(probe_text)], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        z = model.blocks[0].ln1(model.pos_enc(model.tok_emb(probe_idx)))
        attn0 = model.blocks[0].attn
        B, T, D = z.shape
        qkv = attn0.proj_qkv(z).view(B, T, 3, attn0.num_heads, attn0.d_head)
        Qp, Kp, Vp = (t.transpose(1, 2) for t in qkv.unbind(dim=2))
        mask = torch.tril(torch.ones(T, T, device=z.device))
        _, _, Ap = scaled_dot_product_attention(Qp, Kp, Vp, mask=mask)
    n_show = min(4, attn0.num_heads)
    fig, axes = plt.subplots(1, n_show, figsize=(3.2 * n_show, 3.2))
    for h in range(n_show):
        ax = axes[h] if n_show > 1 else axes
        ax.imshow(Ap[0, h].cpu(), cmap="Blues", vmin=0, vmax=1)
        ax.set_title(f"head {h}"); ax.set_xticks(range(T)); ax.set_yticks(range(T))
        ax.set_xticklabels(list(probe_text), rotation=90, fontsize=7)
        ax.set_yticklabels(list(probe_text), fontsize=7)
    fig.suptitle("First-block attention on a real prompt")
    fig.tight_layout(); plt.show()


## Final Checks

In [ ]:
print("=== Final Checks ===")
assert torch.allclose(A_m[0], torch.tensor([1., 0., 0.]))
print("[OK] causal mask verified")
if CKPT_PATH.exists():
    _, loss_check = model(*get_batch("val", model_cfg.block_size, 8))
    assert torch.isfinite(loss_check)
    print(f"[OK] forward pass + loss (val sample: {loss_check.item():.4f})")
assert SEED is not None and model_cfg.vocab_size is not None
print("[OK] config + seed")
assert CKPT_PATH.exists()
print(f"[OK] checkpoint: {CKPT_PATH}")
run_records = sorted(RUN_DIR.glob("run_*.json"))
assert run_records; print(f"[OK] run record: {run_records[-1]}")
print("[OK] sampling gallery above")
print("\nall checks passed")


## Interpretation

*(fill in after the real run)*

- **Training dynamics**: final train/val loss and whether the gap suggests over- or
  underfitting given the corpus size.
- **Samples**: does greedy decoding fall into repetition loops? Does lowering
  temperature from 1.0 noticeably improve local coherence?
- **The FOMC-voice angle**: this model only ever sees statements and minutes through
  Chair Powell's final meeting. Its samples are a snapshot of the outgoing
  communication style — shortly afterward, the new chair's public remarks moved
  toward shorter statements with forward guidance explicitly withdrawn. A natural
  follow-up (not attempted here) would be a second model trained on the new era, to
  compare the two "voices" directly.

## Limitations and Future Work

- Small model, small budget, on purpose — no hyperparameter search, no architecture
  variants, no fine-tuning or RLHF, no deployment.
- Character-level rather than token-level; a lightweight tokenizer would likely
  improve sample quality at the cost of more moving parts.
- FOMC minutes before ~2010 aren't in this corpus (they live under a different,
  uncrawled URL scheme on the Fed's site) — noted in the corpus manifest, not a
  silent gap.
- No comparison model trained on the post-transition communication style yet — see
  Interpretation above.